# Establishment Scoring

This notebook applies the two models saved in `./05_modelling.ipynb` - a balanced classifier and an unweighted, calibrated risk scorer - to the full London establishment dataset, producing a single scored table ready to feed the **Snowflake** and **Tableau** stages of this project.

Four artefacts from the modelling notebook are loaded: 
- the classifier model (`fhrs_classifier.joblib`)
- the risk scorer model (`fhrs_risk_scorer.joblib`)
- the exact feature column order used at training time (`feature_columns.json`)
- the business-type grouping applied before encoding (`business_type_grouping.json`)

Establishments without a usable `income_decile` (~13% of the dataset - see `./03_ons_join.ipynb`) cannot be scored by either model. These are retained in the output with an explicit `scored = False` flag rather than being silently dropped.  

### Load and check saved artefacts

First I will load the required artefacts stored in `../models/` to memory, and then check to ensure they have loaded correctly:

In [1]:
# load and inspect saved artefacts

import joblib
import json

classifier = joblib.load("../models/fhrs_classifier.joblib")
risk_scorer = joblib.load("../models/fhrs_risk_scorer.joblib")

with open("../models/feature_columns.json") as f:
    feature_columns = json.load(f)

with open("../models/business_type_grouping.json") as f:
    rare_types = json.load(f)

print(f"Classifier classes: {classifier.classes_}")
print(f"Risk scorer classes: {risk_scorer.classes_}")
print(f"Feature columns: {len(feature_columns)}")
print(f"Rare categories: {rare_types}")

Classifier classes: ['fail' 'pass']
Risk scorer classes: ['fail' 'pass']
Feature columns: 44
Rare categories: ['Distributors/Transporters', 'Farmers/growers', 'Importers/Exporters']


-> expected classes (`fail`/`pass`), feature count (`44`) and rare categories seen confirm successful loading.  

### Load full dataset, define 'scorable' establishments

Next, I'll load in the full London establishments dataset, and apply the model to applicable establishments.  

An important aspect to note here is that 'scorable' **isn't the same population as the training data**. Training required a _gradable_ rating (**0-5**, excluding 'awaiting' or similar) as well as both `income_decile` and `crime_decile` values.  

Since the final model doesn't require the crime-driven deprivation data, only `income_decile` values are needed from the IMD dataset.

As the 'real-world' risk associated with _ungraded_ businesses is potentially higher than those with a graded rating, it is both worthwhile and appropriate to apply the model to these establishments.  

In [2]:
# dataset load & prep

import pandas as pd

# load full dataset & check shape
df = pd.read_csv(
    "../data/processed/fsa_london_establishments_with_domains.csv",
    dtype={"PostCode": str, "postcode_tier": str, "lsoa21cd": str, "RightToReply": str}
)
print(df.shape)

# flag scorable rows & check count
df["scored"] = df["income_decile"].notna()
print(df["scored"].value_counts())

(81217, 40)
scored
True     70462
False    10755
Name: count, dtype: int64


-> 81,217 total rows, of which 70,462 are scorable - expected values.

### Rebuild feature matrix

Next, I will rebuild the feature matrix for the scorable subset of the data.

Unlike in the model training stage, I won't use `drop_first=True`, as this drops the category that is alphabetically first among what is present in that _specific_ data. If a new categories are introduced (or a category seen in training is no longer present), the 'dropped' reference column could differ from the training, resulting in a different, misaligned set of columns.   

Furthermore, if a category present in training doesn't appear in this data slice, its dummy column won't be created here - this is likely to throw a shape mismatch error.  

With this in mind, I will encode _without_ `drop_first` (generating _every_ possible dummy column), and will then explicitly **reindex** the result to match `feature_columns` exactly. This lets `pandas` fill in any missing columns with zeros, and drops any columns which are not expected.

This approach makes the alignment _explicit_ and safe.

In [3]:
# rebuild feature set on new scorable data

# extract 'scorable' rows
scorable = df[df["scored"]].copy()

# collapse rare business types to other
scorable["BusinessTypeGrouped"] = scorable["BusinessType"].replace(
    dict.fromkeys(rare_types, "Other") 
        # ensures logic from training is replicated
)

# encode categorical features
encoded = pd.get_dummies(
    # drop_first not used (see markdown cell above)
    scorable[["BusinessTypeGrouped", "LocalAuthorityName", "income_decile"]],
    columns=["BusinessTypeGrouped", "LocalAuthorityName"],
)

# reshape encoded to exactly match `feature_columns` from training
X_score = encoded.reindex(columns=feature_columns, fill_value=0)
    # fill_value=0 populates any empty columns not in encoded

# inspect
print(X_score.shape)
print(X_score.columns.equals(pd.Index(feature_columns)))

(70462, 44)
True


-> (70462, 44) is the expected shame, columns match `feature_columns`

### Generate and check predictions

With the 'scorable' data prepared, I can now apply the two saved models (classifier, risk scorer) to the new scorable subset.

Note that with the **classifier**, I'll use `.predict(X_score)` (for the _rating_ label), and with the **risk scorer**, I'll use `predict_proba(X_score)`.   

Due to `class_weight="balanced"` being applied to `classifier` during training, its probability scores are skewed towards `fail` and are not reflective of the overall picture (whereas `risk_scorer`, the unweighted twin model's are) - hence the use of two models, rather than one.  

Finally, I'll store the prediction values in the original - main - DataFrame `df` for scorable rows only, before checking the result.

In [4]:
# generate predictions froms saved models

# predict and store fail/pass label from weighted model
scorable["predicted_rating"] = classifier.predict(X_score)

# predict fail probability from unweighted model
risk_probs = risk_scorer.predict_proba(X_score) # gets both fail and pass probabilities
fail_col = list(risk_scorer.classes_).index("fail") # extract fail probability only
scorable["risk_score"] = risk_probs[:, fail_col] # store extracted values

# write prediction values back to the main DataFrame
    # non-scorable rows will have NaN
    # .loc[] ensures safe alignment (as indices retained from load step onwards)
df.loc[scorable.index, "predicted_rating"] = scorable["predicted_rating"]
df.loc[scorable.index, "risk_score"] = scorable["risk_score"]

# inspect results
print(df[["scored", "predicted_rating", "risk_score"]].sample(10, random_state=42))
print(df["predicted_rating"].value_counts(dropna=False))
print(df["risk_score"].describe())

       scored predicted_rating  risk_score
26077    True             fail    0.480872
15134    True             fail    0.524111
39953    True             fail    0.455395
14858    True             pass    0.193896
29417    True             fail    0.420535
76443    True             pass    0.261692
54134    True             fail    0.512703
49772   False              NaN         NaN
29933   False              NaN         NaN
61827    True             pass    0.272728
predicted_rating
fail    35391
pass    35071
NaN     10755
Name: count, dtype: int64
count    70462.000000
mean         0.335770
std          0.145452
min          0.023862
25%          0.225044
50%          0.332392
75%          0.443114
max          0.662749
Name: risk_score, dtype: float64


-> notes on the results:

- Rating count (35,391 + 35,071 + 10,755 = 81,217) matches total row count, as expected - nothing lost or duplicated during the `.loc` assignment. 
- A near 50/50 `predicted_rating` split between `fail` and `pass` differs from the observed real-world fail rate of 31-33%, however this difference is expected, as a 'signature' of class-weight balancing. _Notably, the output of this column will be expressed as 'requiring attention' and 'low intervention need' (or similar), rather than a flat 'predicted fail' and 'predicted pass'_.  
- The `risk_score` range was `0.0241` to `0.6616` in training, this full-population run gives `0.0239` to `0.6627` - an encouragingly similar pair of numbers, with the drift accounted for by unseen 'awaiting inspection' or similar ungraded rows.  
- The mean `risk_score` (**0.336**) closely tracks the real-world 31-33% 'fail' rate, reflecting a well-calibrated, _unweighted_ model - again, an encouraging sign.  

As a final sense-check, I'll examine whether both models broadly agree in terms of direction, by comparing `risk_score` distribution by `predicted_rating` grouping:

In [5]:
# sanity check - model alignment
print(df.groupby("predicted_rating")["risk_score"].describe())

                    count      mean       std       min       25%       50%  \
predicted_rating                                                              
fail              35391.0  0.455706  0.084005  0.326807  0.386331  0.443114   
pass              35071.0  0.214739  0.078865  0.023862  0.148328  0.223268   

                       75%       max  
predicted_rating                      
fail              0.521256  0.662749  
pass              0.278612  0.336954  


-> a solid confirmation of model alignment seen (e.g. `risk_score` mean for 'fail' group is `0.456`, 'pass' group is significantly lower at `0.215`), again an encouraging sign.

One detail to note is the overlap between the `min` for `fail` (**0.327**) and the `max` for `pass` (**0.337**) - a handful of establishments sit on the boundary, where the two models don't perfectly align. This is not unexpected, and is due to the different class weighting used during model fitting.

Ulitmately, the two models show strong directional agreement, with a small (~1%) overlap zone reflecting their independently-fitted decision boundaries.

### Trim and save finalised column set

With the predictions applied and checked, I will finally save a 'trimmed' version of the data for export.

All columns from the original dataset are still currently present, however the majority are redundant or not useful for the final Tableau visualisation, so I will drop these columns and keep only useful fields.  

In [6]:
# select and check trimmed dataset

output_columns = [
    "FHRSID", "BusinessName", "BusinessType", "LocalAuthorityName", "PostCode",
    "geocode.latitude", "geocode.longitude",
    "RatingValue", "RatingDate",
    "imd_decile", "income_decile",
    "lsoa_match_type", "postcode_tier",
    "scored", "predicted_rating", "risk_score",
]

final_output = df[output_columns].copy()

# check
print(final_output.shape)
print(final_output["FHRSID"].is_unique)
    # uniqueness check - field will become ID downstream

# inspect
final_output.head()

(81217, 16)
True


,FHRSID,BusinessName,BusinessType,LocalAuthorityName,PostCode,geocode.latitude,geocode.longitude,RatingValue,RatingDate,imd_decile,income_decile,lsoa_match_type,postcode_tier,scored,predicted_rating,risk_score
0,1714830,5 Elms Cafe,Restaurant/Cafe/Canteen,Barking and Dagenham,RM8 3NH,51.554493,0.142421,5,2024-07-13T00:00:00,3.0,2.0,full_match,full,True,fail,0.477602
1,115956,7 TILL 11,Retailers - other,Barking and Dagenham,RM8 3UB,51.558435,0.129392,5,2023-09-15T00:00:00,2.0,2.0,full_match,full,True,fail,0.558580
2,1413686,Aafio Mini Market,Retailers - other,Barking and Dagenham,RM9 4QS,51.534718,0.110646,5,2023-09-02T00:00:00,2.0,2.0,full_match,full,True,fail,0.558580
3,128460,Abbey Children's Centre Day Nursery,Caring Premises,Barking and Dagenham,IG11 8JA,51.541375,0.075250,5,2026-05-27T00:00:00,3.0,3.0,full_match,full,True,pass,0.212153
4,122900,Abbey Kebab & Pizza,Takeaway/sandwich shop,Barking and Dagenham,RM8 2XT,51.550930,0.128666,5,2024-07-13T00:00:00,2.0,2.0,full_match,full,True,fail,0.555756


-> all ok, finally save to disk:

In [7]:
# save to disk

import os

os.makedirs("../outputs", exist_ok=True)
final_output.to_csv("../outputs/fhrs_london_scored.csv", index=False)

print(f"Saved {len(final_output)} rows to outputs/fhrs_london_scored.csv")

Saved 81217 rows to outputs/fhrs_london_scored.csv


### Notebook summary

This notebook applied the two models saved in `05_modelling.ipynb` to the full London establishment dataset, producing a single scored table for the Snowflake and Tableau stages ahead.

**Key numbers:**
- 70,462 of 81,217 establishments (86.8%) were scorable - matching the LSOA join rate from `03_ons_join.ipynb` exactly, since `income_decile` availability is the only constraint applied.
- The remaining 10,755 rows (13.2%) are retained with `scored = False` and null predictions, rather than being dropped.
- `predicted_rating` splits close to 50/50 (`fail` 35,391 / `pass` 35,071) - a known signature of `class_weight="balanced"`, not a real-world estimate. It should be interpreted as a flag for attention, not a literal fail-rate prediction; `risk_score` is the calibrated figure to use for anything quantitative.
- `risk_score` closely reproduces its training-time range (0.024–0.663 vs. 0.024–0.662) and mean (0.336, close to the true ~31–33% population fail rate), and the two models agree directionally (`fail`-predicted mean risk 0.456 vs. `pass`-predicted mean 0.215) - three independent signs the models generalise sensibly to the full population, including establishments (e.g. `AwaitingInspection`) never seen during training.

**Output:** `outputs/fhrs_london_scored.csv` - 81,217 rows, 16 columns, one row per establishment (`FHRSID` confirmed unique), ready for Snowflake ingestion.